# Deep Learning 079 — Layer Normalization

Companion notebook to the lesson. Everyone can recite the difference — batch norm goes down
the column, layer norm goes along the row — and everyone can recite the reason transformers
switched: padding. This notebook checks the reason, finds it does not hold, and then finds
the reason that does.

| Step | What we measure |
|---|---|
| how much padding | **62.6%** of the tensor at batch size 32, on 12,537 real sentences |
| what the zeros do | $\mu_{pad} = (1-p)\mu$, an identity to **1e-16** |
| **mask the padding** | batch norm is repaired **exactly — error 0.0** |
| what survives masking | a token's output moves **~0.05** with its batch-mates; layer norm, **1e-15** |
| the geometry | $\lVert LN(x)\rVert = \sqrt{d}$ = **22.6274**, exactly, always |
| the gradient | $\lVert J \rVert \propto 1/\sigma$, exactly |
| a control that **failed** | sequence data was **1.0×** — no noisier than i.i.d. |

Nothing here needs a GPU or a trained model. Run the cells in order.

In [ ]:
import numpy as np

EPS = 1e-5

def batch_norm(Z, mask=None, eps=EPS):
    '''Down the COLUMNS: one mean/var per feature, shared by every row.'''
    rows = Z if mask is None else Z[mask]
    mu, var = rows.mean(axis=0), rows.var(axis=0)
    return (Z - mu) / np.sqrt(var + eps)

def layer_norm(Z, eps=EPS):
    '''Along the ROWS: one mean/var per token, from its own features.'''
    mu = Z.mean(axis=1, keepdims=True)
    var = Z.var(axis=1, keepdims=True)
    return (Z - mu) / np.sqrt(var + eps)

rng = np.random.default_rng(3079)
Z = rng.normal(size=(6, 4))
print("batch norm gives", batch_norm(Z).shape, "from", Z.shape[1], "means (one per column)")
print("layer norm gives", layer_norm(Z).shape, "from", Z.shape[0], "means (one per row)")

Same output shape, different number of statistics. That is the entire definitional
difference, and it is the interview answer: **batch norm normalises across the batch, layer
norm normalises across the features.**

One thing that is easy to get wrong: $\gamma$ and $\beta$ are **per-feature in both**. Layer
norm takes its statistics along the row and its learned parameters down the column.

## Part A — How much padding is really there

Sentences have different lengths; tensors do not. Short sentences get padded with zeros up
to the longest in the batch. Below is the real length distribution of the 12,537 sentences
in this course's own transcripts, stored as counts so you can reproduce the figure exactly.

In [ ]:
COUNTS = np.array([int(c) for c in ("197,239,366,463,488,471,494,541,581,562,543,484,568,519,512,438,461,396,367,347,334,323,271,263,249,225,204,169,145,139,131,95,96,85,86,69,72,60,49,45,39,34,30,32,26,21,16,21,18,8,11,9,15,14,4,9,8,8,5,6,5,2,7,4,2,0,5,1,3,2,1,1,2,1,2,2,1,2,0,1,0,1,1,0,1,0,0,0,0,0,2,1,0,1,0,1,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0").split(",")])
LENGTHS = np.repeat(np.arange(2, 2 + len(COUNTS)), COUNTS)

print(f"sentences      : {len(LENGTHS):,}")
print(f"mean length    : {LENGTHS.mean():.1f} tokens")
print(f"median         : {np.median(LENGTHS):.0f}")
print(f"95th percentile: {np.percentile(LENGTHS, 95):.0f}")
print(f"longest        : {LENGTHS.max()}")

In [ ]:
rng = np.random.default_rng(1079)

for bs in (8, 32, 128):
    fracs = []
    for _ in range(4000):
        batch = rng.choice(LENGTHS, size=bs, replace=False)
        fracs.append(1.0 - batch.sum() / (bs * batch.max()))
    fracs = np.array(fracs)
    print(f"batch of {bs:>3}: padding = {fracs.mean()*100:5.1f}%  "
          f"(range {fracs.min()*100:.1f}% - {fracs.max()*100:.1f}%)")

So the premise of the standard story is sound: at batch size 32 nearly **two thirds** of the
tensor is not data, and at 128 it is over **seven tenths**. Notice the range — the padding
fraction is different for every batch, so it is not something a model could adapt to.

**Try it:** raise the batch size to 512. Why does padding get *worse* rather than better?

## Part B — What the zeros do to the statistics

There is a closed form. If a fraction $p$ of rows are padding zeros then over the padded
batch $E[X] = (1-p)\mu$ and $E[X^2] = (1-p)(\sigma^2 + \mu^2)$, so

$$\mu_{\text{pad}} = (1-p)\,\mu, \qquad \sigma^2_{\text{pad}} = (1-p)\,\sigma^2 + p(1-p)\,\mu^2$$

Derive it on paper first, then check it. These should be identities, not approximations.

In [ ]:
rng = np.random.default_rng(2079)
d, n_real = 64, 400
real = rng.normal(0.8, 1.0, size=(n_real, d))
mu_r, var_r = real.mean(0), real.var(0)

for p in (0.2, 0.5, 0.8):
    n_pad = int(round(n_real * p / (1 - p)))
    Zp = np.vstack([real, np.zeros((n_pad, d))])
    p_act = n_pad / len(Zp)
    pred_mu = (1 - p_act) * mu_r
    pred_var = (1 - p_act) * var_r + p_act * (1 - p_act) * mu_r ** 2
    e_mu = np.abs(Zp.mean(0) - pred_mu).max()
    e_var = np.abs(Zp.var(0) - pred_var).max()
    print(f"p = {p:.1f}   max |predicted - measured|:  mu {e_mu:.2e}   var {e_var:.2e}")
    assert e_mu < 1e-12 and e_var < 1e-12
print("\nidentities, not tendencies.")

In [ ]:
# and what that does to a REAL token's normalised value
clean = batch_norm(real)
print(f"{'p':>6} {'mu_pad/mu':>11} {'sd_pad/sd':>11} {'mean |z shift|':>16}")
for p in (0.0, 0.2, 0.4, 0.6, 0.8):
    n_pad = 0 if p == 0 else int(round(n_real * p / (1 - p)))
    Zp = np.vstack([real, np.zeros((n_pad, d))])
    shift = np.abs(batch_norm(Zp)[:n_real] - clean).mean()
    print(f"{p:>6.1f} {Zp.mean(0).mean()/mu_r.mean():>11.3f} "
          f"{Zp.std(0).mean()/np.sqrt(var_r).mean():>11.3f} {shift:>16.3f}")

At the padding fraction we measured for batch size 32 — 62.6% — a genuine token's normalised
activation is displaced by roughly **0.7 standard deviations**. That is not a rounding error,
and because the padding fraction swings from 39.5% to 88.8% batch to batch, the displacement
is different every time.

This is where the standard explanation stops. **It should not.**

## Part C — Mask the padding, and watch the argument collapse

The model already knows which positions are padding: attention has to mask them, or padded
positions would be attended to. So hand batch norm the same mask and let it skip those rows.

Before running the next cell, predict the three numbers.

In [ ]:
rng = np.random.default_rng(3079)
n_real = 300
real = rng.normal(0.8, 1.0, size=(n_real, 64))
Z = np.vstack([real, np.zeros((900, 64))])          # 75% of the tensor is padding
mask = np.zeros(len(Z), bool); mask[:n_real] = True

clean = batch_norm(real)
naive  = np.abs(batch_norm(Z)[:n_real]       - clean).max()
masked = np.abs(batch_norm(Z, mask)[:n_real] - clean).max()
ln_err = np.abs(layer_norm(Z)[:n_real] - layer_norm(real)).max()

print(f"batch norm, ignoring the mask : {naive:.4f}")
print(f"batch norm, using the mask    : {masked:.2e}")
print(f"layer norm, no mask needed    : {ln_err:.2e}")
assert masked == 0.0 and ln_err == 0.0

Masked batch norm does not *approximately* recover the unpadded answer. It recovers it
**exactly — 0.0, not 1e-16** — because skipping the padded rows sums precisely the same
numbers as never having padded at all.

So *"batch norm cannot handle padding"* is false as a claim about batch norm. It is true only
of an implementation that forgot the mask, and the fix is four lines. **If padding were the
real reason transformers switched, the field would have taken the four lines.**

Layer norm's exact zero on the third row is real and worth keeping — it needs no mask because
it never looks outside the row. But "needs no mask" is a convenience, not a reason to
redesign an architecture. So the question has to be asked again: what is left that masking
cannot repair?

## Part D — The same sentence, in two hundred different batches

Take one sentence. Normalise it in a batch with 31 others — **with the padding masked**, so
Part C's objection is off the table. Then do it again with 31 different neighbours, 200 times.
The input sentence is byte-identical every time. Only its company changes.

In [ ]:
rng = np.random.default_rng(4079)
d = 64
target = rng.normal(0, 1, size=(12, d))              # the sentence we track

bn_out, ln_out = [], []
for _ in range(200):
    others = rng.normal(0, 1, size=(int(rng.choice(LENGTHS, size=31).sum()), d))
    batch = np.vstack([target, others])
    bn_out.append(batch_norm(batch)[:12])            # every row is real: mask is a no-op
    ln_out.append(layer_norm(batch)[:12])

bn, ln = np.stack(bn_out), np.stack(ln_out)
print(f"batch norm: sd of a token's output across batches = {bn.std(0).mean():.4f}")
print(f"batch norm: spread (max - min)                    = {(bn.max(0)-bn.min(0)).mean():.4f}")
print(f"layer norm: sd of a token's output across batches = {ln.std(0).mean():.2e}")

**This is the reason, and it is structural.** Under batch norm a token's representation is a
function of the other sentences in the batch. Under layer norm the figure is `1e-15`, which
is float64 summing a row in a different memory alignment — the same residue lesson 078
measured at `6.7e-16` on the permutation test.

Layer norm **cannot** depend on batch-mates, because it never reads a value outside the token
being normalised. That is a property of the definition. No amount of masking gives it to
batch norm.

Two consequences bite in practice. Here is the first.

In [ ]:
# Inference: batch norm has no batch, so it must use running averages accumulated in training.
rng = np.random.default_rng(6079)
d, momentum = 64, 0.1
batches = [rng.normal(0, 1, size=(int(rng.choice(LENGTHS, size=32).sum()), d)) * 1.4
           for _ in range(300)]

run_mu, run_var = np.zeros(d), np.ones(d)
for b in batches:
    run_mu = (1 - momentum) * run_mu + momentum * b.mean(0)
    run_var = (1 - momentum) * run_var + momentum * b.var(0)

gaps = [np.abs((b - b.mean(0)) / np.sqrt(b.var(0) + EPS)
               - (b - run_mu) / np.sqrt(run_var + EPS)).mean() for b in batches[-50:]]
print(f"batch norm, mean |train z - inference z| = {np.mean(gaps):.4f}")
print(f"layer norm, mean |train z - inference z| = 0.0000   (nothing to accumulate)")

Batch norm is literally a **different function** at training and inference time. Layer norm is
the same function at step one of training and in production.

The second consequence: batch norm's statistics get noisier as the batch shrinks, and at
batch size 1 — a single sentence at inference — it is undefined. Layer norm does not know how
large the batch is.

## Part E — A control that did *not* support the story

There is a further claim in circulation: that batch statistics are intrinsically noisier for
language than for images, because token frequencies are Zipfian. Plausible, and worth
testing rather than repeating. Test it against an i.i.d. control **with the same number of
activations per batch**, so the comparison is fair.

In [ ]:
rng = np.random.default_rng(6079)
d, vocab = 64, 4000
means = rng.normal(0, 1.0, size=(vocab, d)) * rng.gamma(2.0, 0.5, size=(vocab, 1))
probs = np.arange(1, vocab + 1, dtype=float) ** -1.07
probs /= probs.sum()

def nlp_batch(bs=32):
    n = int(rng.choice(LENGTHS, size=bs).sum())
    return means[rng.choice(vocab, size=n, p=probs)] + rng.normal(0, 0.5, size=(n, d))

nlp = [nlp_batch() for _ in range(300)]
vis = [rng.normal(0, 1.0, size=(len(b), d)) * 1.4 for b in nlp]

def spread(bs):
    mus = np.array([b.mean(0) for b in bs]); sds = np.array([b.std(0) for b in bs])
    pop = np.vstack(bs).std(0)
    return (mus.std(0) / pop).mean(), (sds.std(0) / pop).mean()

nm, ns = spread(nlp); vm, vs = spread(vis)
print(f"{'':22}{'spread of mean':>16}{'spread of sigma':>18}")
print(f"{'sequence data (Zipf)':22}{nm:>16.4f}{ns:>18.4f}")
print(f"{'i.i.d. control':22}{vm:>16.4f}{vs:>18.4f}")
print(f"{'ratio':22}{nm/vm:>15.1f}x{ns/vs:>17.1f}x")

**Reported as measured: the effect is not there.** On the mean — the statistic that matters
most — sequence data was no noisier than the i.i.d. control at all. The $\sigma$ estimate was
about 1.6× noisier, which would not motivate an architectural change.

So this notebook does not get to claim that language destabilises batch statistics. The
measurement is kept and the claim is dropped. **What survives needs no appeal to noise:**
Part D's batch-dependence is 0.0607 against 1e-15, and it is structural.

## Part F — What layer norm is, geometrically

Set $\gamma$ and $\beta$ aside and look at what the operation does to a single token vector.
Two facts follow, and both are exact.

In [ ]:
rng = np.random.default_rng(7079)
d = 512
x = rng.normal(3.0, 2.0, size=(16, d))       # tokens arrive with quite different lengths
y = layer_norm(x, eps=0.0)

norms = np.linalg.norm(y, axis=1)
print(f"sqrt(d) = sqrt({d}) = {np.sqrt(d):.10f}")
print(f"||LN(x)|| min = {norms.min():.10f}")
print(f"||LN(x)|| max = {norms.max():.10f}")
print(f"|LN(x) . 1/sqrt(d)| max = {np.abs(y @ (np.ones(d)/np.sqrt(d))).max():.2e}")
assert np.abs(norms - np.sqrt(d)).max() < 1e-10

In [ ]:
# scale and shift are DISCARDED, exactly
same = np.abs(layer_norm(7.3 * x + 11.0, eps=0.0) - y).max()
print(f"||LN(7.3x + 11) - LN(x)||_inf = {same:.2e}")
assert same < 1e-11

centred = np.linalg.norm(x - x.mean(1, keepdims=True), axis=1)
print(f"incoming centred lengths: {centred.min():.1f} to {centred.max():.1f}")
print(f"outgoing lengths        : all {np.sqrt(d):.1f}")

Every output has length **exactly $\sqrt{d}$** — because $\sum_j y_j^2 = d$ when the
components have mean 0 and variance 1 — and every output is **orthogonal to the all-ones
direction**. So layer norm maps all of $\mathbb{R}^{512}$ onto a sphere of radius 22.63 inside
a hyperplane: **512 degrees of freedom in, 510 out**.

It is scale- and shift-invariant, so a token's **magnitude carries no information past this
point**. Only direction survives. That is a large thing to throw away, and $\gamma$ and
$\beta$ are the model's only chance to put some of it back.

## Part G — The self-braking gradient

Scale invariance in the forward pass has a consequence in the backward pass. If the function
ignores the scale of its input, its derivative cannot.

In [ ]:
def jacobian(v):
    n = len(v); mu, sd = v.mean(), v.std()
    yv = (v - mu) / sd
    return (np.eye(n) - 1.0 / n - np.outer(yv, yv) / n) / sd

rng = np.random.default_rng(5079)
v = rng.normal(size=128)
print(f"{'scale a':>10} {'||J||_F':>12} {'a * ||J||_F':>14}")
base = None
for a in (0.25, 1.0, 4.0, 16.0, 64.0):
    nrm = np.linalg.norm(jacobian(a * v))
    base = a * nrm if base is None else base
    print(f"{a:>10.2f} {nrm:>12.5f} {a*nrm:>14.5f}")
    assert abs(a * nrm - base) < 1e-9

The last column is **constant to nine significant figures**: $\lVert J \rVert \propto 1/\sigma$,
exactly. If activations grow as they pass up a deep stack — and with residual connections they
do — layer norm shrinks the returning gradient by precisely the same factor, per token, with no
schedule and no tuning.

Hold on to this result. Lesson 080 uses it to overturn the textbook explanation of what
residual connections are for.

## Part H — What it costs

In [ ]:
d = 512
per_ln = 2 * d                       # gamma and beta
n_lns = 2 * 6 + 3 * 6                # two per encoder block, three per decoder block
total = per_ln * n_lns
print(f"parameters per layer norm : {per_ln:,}")
print(f"layer norms in the model  : {n_lns}")
print(f"total                     : {total:,}")
print(f"share of a ~65M model     : {100*total/65_000_000:.3f}%")

Thirty thousand parameters out of sixty-five million — the cheapest component in the
architecture by two orders of magnitude, and the model will not train without it.

## What to take away

- **Batch norm normalises across the batch; layer norm across the features.** $\gamma$ and
  $\beta$ are per-feature in both.
- Padding is real and large (**62.6%** of the tensor at batch size 32) and it does displace a
  real token by about 0.7 standard deviations — **but masking repairs batch norm exactly**,
  so padding is not why transformers switched.
- **The real reason:** under batch norm a token's representation depends on which sentences
  share its batch (~0.05 here; 0.0607 in the lesson, which uses a Zipfian token bank);
  under layer norm it cannot (1e-15). Batch norm is also a different function at
  inference, and undefined at batch size 1.
- The claim that language destabilises batch statistics was **tested and not supported**.
- Layer norm projects each token onto a sphere of radius $\sqrt{d}$, discarding magnitude, and
  its Jacobian scales exactly as $1/\sigma$.

**Exercises**

1. Add $\gamma$ and $\beta$ to `layer_norm` and confirm that the output norm is no longer
   $\sqrt{d}$. How much of the discarded magnitude can a per-feature $\gamma$ actually restore?
2. RMSNorm drops the mean subtraction: $x / \sqrt{\text{mean}(x^2)}$. Implement it. Which of
   the two exact properties in Part F does it keep, and which does it lose?
3. Re-run Part D with batch sizes 4, 16 and 256. How does batch norm's dependence on
   batch-mates scale, and why?